# Notebook 1 — Data Pull

**Purpose:** pull daily price/rate data from two sources — Yahoo Finance and FRED — and save each series as a CSV into your Google Drive. Notebook 2 will use these files for the regime analysis.

Run the cells in order, top to bottom. Check the log output under each cell before moving to the next.

## Setup

Installs libraries, connects to your Google Drive. Run this first, every time.

In [ ]:
# --- THE COMPLETE IGNITION SWITCH ---

# 1. System Install (The "Plumbing")
!pip install yfinance pandas_datareader -q

# 2. Python Imports (The "Tools")
from google.colab import drive
import time
import os
import pandas as pd
import yfinance as yf
import pandas_datareader.data as web

# 3. Drive Connection (The "Storage")
drive.mount('/content/drive')

# 4. Shared settings
start_date = "2003-12-31"
end_date = pd.Timestamp.today().strftime("%Y-%m-%d")

data_dir = '/content/drive/MyDrive/MSFin_DA_Bootcamp/data'
os.makedirs(data_dir, exist_ok=True)

print("✅ System Ready: Libraries installed, Drive mounted, data folder confirmed.")
print(f"Data will be saved to: {data_dir}")

## Yahoo Finance pull

Pulls daily price data for each ticker in the list below and saves each as its own CSV.

**To add a ticker:** just add it to the `tickers` list (e.g. `"QQQ"`) and re-run this cell.

Columns saved: `Date, Open, High, Low, Close, Volume` — the natural order yfinance returns.

In [ ]:
tickers = ["RSP", "SPY", "TLT"]

for ticker in tickers:
    try:
        prices = yf.download(ticker, start=start_date, end=end_date, progress=False)

        if prices.empty:
            print(f"⚠️  {ticker}: no data returned — check the ticker symbol and try again.")
            continue

        # yfinance returns a MultiIndex column header when passed a single ticker string
        # in newer versions; flatten it so the CSV has plain column names
        if isinstance(prices.columns, pd.MultiIndex):
            prices.columns = prices.columns.get_level_values(0)

        prices_flat = prices.reset_index()

        file_path = f"{data_dir}/{ticker}_data.csv"
        prices_flat.to_csv(file_path, index=False)

        print(f"✅ {ticker}: {len(prices_flat)} rows saved to {ticker}_data.csv")

    except Exception as e:
        print(f"⚠️  {ticker}: failed — {e}")

## FRED pull

Pulls daily levels for VIX and the Treasury curve from FRED (no API key required) and saves each as its own CSV.

This cell isn't meant to be edited — the series list is fixed for now.

Columns saved: `date, close`.

In [ ]:
# FRED series code -> friendly name used in the saved filename
fred_series = {
    "VIXCLS": "VIX",
    "DGS3MO": "UST3MO",
    "DGS1":   "UST1YR",
    "DGS2":   "UST2YR",
    "DGS5":   "UST5YR",
    "DGS10":  "UST10YR",
    "DGS20":  "UST20YR",
    "DGS30":  "UST30YR",
}

pause_seconds = 1.5  # fixed pause between requests so FRED doesn't rate-limit the pulls

for code, friendly_name in fred_series.items():
    try:
        series = web.DataReader(code, "fred", start_date, end_date)

        if series.empty:
            print(f"⚠️  {friendly_name} ({code}): no data returned.")
            time.sleep(pause_seconds)
            continue

        series = series.reset_index()
        series.columns = ["date", "close"]

        file_path = f"{data_dir}/{friendly_name}_data.csv"
        series.to_csv(file_path, index=False)

        print(f"✅ {friendly_name} ({code}): {len(series)} rows saved to {friendly_name}_data.csv")

    except Exception as e:
        print(f"⚠️  {friendly_name} ({code}): failed — {e}")

    time.sleep(pause_seconds)